In [ ]:
import os, glob
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.stats import spearmanr

In [ ]:
from pathlib import Path

# Project root is one level above the notebook folder.
# Use Path.cwd() instead if the notebook is directly in the project root.
PROJECT_ROOT = Path.cwd().parent

# Main input folders
DISTRIBUTIONS_DIR = PROJECT_ROOT / "distributions4piton"
SUMMARY_STATS_DIR = PROJECT_ROOT / "summary_stats"
SUMMARY_COMBINED_DIR = PROJECT_ROOT / "summary_stats_combined"

# Main output folders
FINAL_RESULT_DIR = PROJECT_ROOT / "final_result"
COVARIANCE_DIR = FINAL_RESULT_DIR / "covariance_matrix"

# Create output folders if they do not exist
FINAL_RESULT_DIR.mkdir(parents=True, exist_ok=True)
COVARIANCE_DIR.mkdir(parents=True, exist_ok=True)

# ICC

In [ ]:
RXN_MATRIX_PATH = "../summary_stats_combined/FAC_by_MAR_median_matrix.csv" # common_rxns from FAC_by_MAR_median_matrix.csv - columns
rxn_universe = pd.read_csv(RXN_MATRIX_PATH, nrows=0).columns
common_rxns = pd.Index(rxn_universe).sort_values()
print(f"Reactions from matrix columns: {len(common_rxns)}")
common_rxns = common_rxns.drop('FAC')
common_rxns = common_rxns.drop('Unnamed: 0')

In [ ]:
# General output directory
OUT_DIR = FINAL_RESULT_DIR
# For full sampled flux distribution files
IN_DIR = DISTRIBUTIONS_DIR

In [ ]:
# Paths
SUM_DIR = SUMMARY_STATS_DIR
FILE_GLOB = "FAC*_summary.csv"

# Load summary files
paths = sorted(glob.glob(os.path.join(SUM_DIR, FILE_GLOB)))
assert len(paths) > 1, "Need multiple individuals"

# Build MU / VAR matrices
R = len(common_rxns)
N = len(paths)

MU  = np.zeros((N, R), dtype=float)
VAR = np.zeros((N, R), dtype=float)
ids = []

for i, path in enumerate(paths):
    facid = os.path.basename(path).replace("_summary.csv", "")
    df = pd.read_csv(path, index_col=0)

    # align + filter to common reactions
    df = df.loc[common_rxns]

    MU[i, :]  = df["mean"].to_numpy(float)
    VAR[i, :] = df["std"].to_numpy(float) ** 2

    ids.append(facid)

ids = np.array(ids)

print(f"Loaded {N} individuals × {R} reactions (filtered)")

In [ ]:
# Use reaction identifiers corresponding to MU/VAR columns
rxn_index = common_rxns

# --- Variance components ---

# Between-individual variance: variance of mean reaction values across individuals
between_var = MU.var(axis=0, ddof=1)   # shape: (n_reactions,)

# Within-individual variance: average variance within individuals
within_var = VAR.mean(axis=0)          # shape: (n_reactions,)

# --- Intraclass Correlation Coefficient (ICC) ---

# ICC = proportion of total variance explained by between-individual differences
icc = between_var / (between_var + within_var)

# Clip values to valid range [0, 1] for numerical stability
icc = np.clip(icc, 0.0, 1.0)

# --- Build results table ---

reaction_icc = pd.DataFrame({
    "reaction": rxn_index.to_numpy(),     # reaction identifiers
    "ICC": icc,                           # reliability / stability metric
    "between_var_of_mean": between_var,   # between-individual variance
    "within_var_mean": within_var         # within-individual variance
}).set_index("reaction")

# Sort reactions by highest ICC (most stable/reproducible)
reaction_icc = reaction_icc.sort_values("ICC", ascending=False)

In [ ]:
out_path = os.path.join(OUT_DIR, "reaction_ICC.csv")
reaction_icc.to_csv(out_path)

print("Saved per-reaction ICC to:", out_path)
print("Top 10 reactions by ICC:")
print(reaction_icc.head(10))
print("\nSummary:")
print(reaction_icc["ICC"].describe())
print("Fraction ICC > 0.5:", (reaction_icc["ICC"] > 0.5).mean()) # just to check

## ICC plot

In [ ]:
# Read the saved ICC data for further analysis
ICC_PATH = FINAL_RESULT_DIR / "reaction_ICC.csv"
ICC_data = pd.read_csv(ICC_PATH)

In [ ]:
# Create the Distribution Plot (Histogram)
plt.figure(figsize=(8, 5))

sns.histplot(data=ICC_data, x="ICC", bins=50, kde=False, color='cornflowerblue')

plt.xlabel("ICC", fontsize=16)
plt.ylabel("Number of Reactions", fontsize=16)
plt.show()

# Correlation/covariance matrix

In [ ]:
# =========================
# Covariance / volume calculation
# =========================

# Input folder containing full sampled flux distributions.
IN_DIR = DISTRIBUTIONS_DIR

# Output folder for covariance-derived results.
OUT_DIR = COVARIANCE_DIR
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Full sampled flux files.
FILE_GLOB = "FAC*_full.csv"
paths = sorted(glob.glob(str(IN_DIR / FILE_GLOB)))

if len(paths) == 0:
    raise FileNotFoundError(f"No files matching {FILE_GLOB} found in {IN_DIR}")

# Use the common reaction set defined earlier in the notebook.
rxns_cov = common_rxns.sort_values()

# Number of principal dimensions used for the volume proxy.
PC_DIMS = 50
EPS = 1e-12

# Folder for saving covariance matrices.
COV_DIR = OUT_DIR / "cov_matrices_npz"
COV_DIR.mkdir(parents=True, exist_ok=True)

rows = []

for path in paths:
    facid = path.split("\\")[-1].replace("_full.csv", "")

    # Load sampled flux distribution for one mouse.
    df = pd.read_csv(path)

    # Keep only common reaction columns.
    available_rxns = [r for r in rxns_cov if r in df.columns]
    X = df[available_rxns].copy()

    # Convert to numeric and remove invalid values.
    X = X.apply(pd.to_numeric, errors="coerce")
    X = X.replace([np.inf, -np.inf], np.nan).dropna(axis=0)

    if X.shape[0] < 2 or X.shape[1] < 2:
        print(f"Skipping {facid}: not enough valid data")
        continue

    # Center flux samples before covariance calculation.
    X_centered = X - X.mean(axis=0)

    # Covariance matrix across reactions.
    cov = np.cov(X_centered.to_numpy(), rowvar=False)

    # Numerical cleanup.
    cov = np.nan_to_num(cov, nan=0.0, posinf=0.0, neginf=0.0)

    # Save covariance matrix for this mouse.
    cov_path = COV_DIR / f"{facid}_cov.npz"
    np.savez_compressed(
        cov_path,
        cov=cov,
        reactions=np.array(available_rxns, dtype=object)
    )

    # Eigenvalues define spread of sampled flux space.
    eigvals = np.linalg.eigvalsh(cov)
    eigvals = np.clip(eigvals, EPS, None)
    eigvals_sorted = np.sort(eigvals)[::-1]

    # Trace = total flux variance.
    trace_cov = float(np.trace(cov))

    # Log-volume proxy from the top principal dimensions.
    k_used = min(PC_DIMS, len(eigvals_sorted))
    log_volume_topk = float(np.sum(np.log(eigvals_sorted[:k_used])))

    rows.append({
        "id": facid,
        "n_samples": int(X.shape[0]),
        "n_reactions": int(X.shape[1]),
        "trace_cov": trace_cov,
        "log_volume_topk": log_volume_topk,
        "k_used": int(k_used),
    })

# Save covariance / volume summary table.
cov_summary = pd.DataFrame(rows).sort_values("id")

summary_path = OUT_DIR / "feasible_space_cov_volume_proxies_50.csv"
cov_summary.to_csv(summary_path, index=False)

print("\nSaved summary:", summary_path)
print("Saved covariance matrices in:", COV_DIR)
print(cov_summary.head())

## plot trace and volume

In [ ]:
# =========================
# Volume vs total flux variance
# =========================

# Load covariance / volume summary generated in the previous cell
SUMMARY_PATH = COVARIANCE_DIR / "feasible_space_cov_volume_proxies_50.csv"
df = pd.read_csv(SUMMARY_PATH)

# Keep only rows needed for the manuscript-relevant geometry plot
df = df.dropna(subset=["log_volume_topk", "trace_cov"])

# Create the same manuscript-relevant scatter plot as before
fig, ax = plt.subplots(figsize=(6, 5), dpi=200)

sns.scatterplot(
    data=df,
    x="log_volume_topk",
    y="trace_cov",
    alpha=0.6,
    edgecolor=None,
    ax=ax
)

# Same y-axis scaling as the original plot
ax.set_yscale("log")

# Axis labels
ax.set_xlabel("Sampled flux space volume")
ax.set_ylabel("Total flux variance")

# Clean layout
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()
plt.show()

# Visualization of volume

In [ ]:
# Load summary data and remove rows with missing values
SUMMARY_PATH = COVARIANCE_DIR / "feasible_space_cov_volume_proxies_50.csv"
df = pd.read_csv(SUMMARY_PATH).dropna(subset=["id", "log_volume_topk"])

# Sort mice by volume (descending) and assign sequential index
df = df.sort_values("log_volume_topk", ascending=False).reset_index(drop=True)
df["mouse_num"] = range(1, len(df) + 1)

# Create figure
fig, ax = plt.subplots(figsize=(16, 8), dpi=200)

# Plot volume per mouse
ax.plot(
    df["mouse_num"],
    df["log_volume_topk"],
    marker="o",
    markersize=3,
    linewidth=1
)

# Axis labels
ax.set_xlabel("Mouse", fontsize=18)
ax.set_ylabel("Sampled flux space volume", fontsize=18)

# Define x-axis ticks (1, then every 10)
ticks = [1] + list(range(10, len(df) + 1, 10))
ticks = sorted(set(ticks))

ax.set_xticks(ticks)
ax.set_xticklabels(ticks, fontsize=12)

# Style adjustments
ax.tick_params(axis="y", labelsize=14)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.grid(axis="y", alpha=0.2)

# Ensure full range is visible
ax.set_xlim(1, len(df))

# Final layout and display
plt.tight_layout()
plt.show()

# Correlation analysis

In [ ]:
# Load volume summary
vol = pd.read_csv(SUMMARY_PATH)

vol_sorted = vol.sort_values("log_volume_topk", ascending=False)

top_mice = vol_sorted.head(5)["id"].tolist()
bottom_mice = vol_sorted.tail(5)["id"].tolist()

print("Top volume mice:", top_mice)
print("Bottom volume mice:", bottom_mice)


In [ ]:
# Project root is one level above the notebook folder.
# Use Path.cwd() instead if the notebook is directly in the project root.
PROJECT_ROOT = Path.cwd().parent

# Input: covariance/volume summary generated earlier.
COVARIANCE_DIR = PROJECT_ROOT / "final_result" / "covariance_matrix"
VOLUME_SUMMARY_PATH = COVARIANCE_DIR / "feasible_space_cov_volume_proxies_50.csv"

# Output folder for the correlation result.
COVARIANCE_DIR.mkdir(parents=True, exist_ok=True)

# Load covariance-derived geometry measures.
df = pd.read_csv(VOLUME_SUMMARY_PATH)

# Required columns for the manuscript correlation.
cols_needed = ["trace_cov", "log_volume_topk"]
missing = [c for c in cols_needed if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

# Spearman correlation reported in the manuscript:
# trace_cov = total flux variance
# log_volume_topk = volume proxy of the sampled flux space
rho, pval = spearmanr(
    df["trace_cov"],
    df["log_volume_topk"],
    nan_policy="omit"
)

# Save result.
corr_result = pd.DataFrame({
    "x": ["trace_cov"],
    "y": ["log_volume_topk"],
    "spearman_rho": [rho],
    "p_value": [pval],
    "n_used": [df[cols_needed].dropna().shape[0]]
})

out_path = COVARIANCE_DIR / "trace_volume_spearman.csv"
corr_result.to_csv(out_path, index=False)

print(f"Spearman rho trace_cov vs log_volume_topk = {rho:.3f}")
print(f"p-value = {pval:.3e}")
print(f"Saved: {out_path}")